In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
import tensorflow as tf
import healpy as hp

from deepsphere import HealpyGCNN

from deep_lss.models.base_model import BaseModel

In [3]:
tf.print("test")

Metal device set to: Apple M1 Max

systemMemory: 32.00 GB
maxCacheSize: 10.67 GB

test


2023-02-21 15:21:06.303980: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-02-21 15:21:06.304621: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


### constants

In [4]:
dir_checkpoint = "./base_checkpoint"

# checkpointing tests

### setup a base model to be saved

In [6]:
network = tf.keras.Sequential([
    tf.keras.layers.InputLayer((None, 4)),
    tf.keras.layers.Dense(4)
    ])

model = BaseModel(network, (None, 2), dir_checkpoint=dir_checkpoint)
model.train_step.assign(10)
X = tf.random.uniform((10, 4), seed=1)
y = tf.random.uniform((10, 4), seed=2)
for _ in range(10):
    model.base_train_step(X, tf.keras.losses.mean_squared_error, y)

print(model.network.weights, "\n\n\n")
print(model.train_step, "\n\n\n")
print(model.optimizer.variables(), "\n\n\n")
model.save_model()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, None, 4)           20        
                                                                 
Total params: 20
Trainable params: 20
Non-trainable params: 0
_________________________________________________________________
23-02-21 15:21:25 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
23-02-21 15:21:25 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
23-02-21 15:21:25 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
23-02-21 15:21:25 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
23-02-21 15:21:25 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
23-02-21 15:21:25 base_model.p WAR   Performing a base_train_step in python 

### load that model
the variables should be identical

In [7]:
network = tf.keras.Sequential([
    tf.keras.layers.InputLayer((None, 4)),
    tf.keras.layers.Dense(4)
    ])

model_1 = BaseModel(network, (None, 2), dir_checkpoint=dir_checkpoint, restore_from_checkpoint=True)
print(model_1.network.weights, "\n\n\n")
print(model_1.train_step, "\n\n\n")
print(model_1.optimizer.variables(), "\n\n\n")

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, None, 4)           20        
                                                                 
Total params: 20
Trainable params: 20
Non-trainable params: 0
_________________________________________________________________
23-02-21 15:21:28 base_model.p INF   Network successfully restored from checkpoint ./base_checkpoint/ckpt-1. 
[<tf.Variable 'dense_2/kernel:0' shape=(4, 4) dtype=float32, numpy=
array([[-0.32454836,  0.3610835 ,  0.3222433 ,  0.21276914],
       [ 0.04707031, -0.29088658, -0.18150042,  0.71360534],
       [ 0.8523498 ,  0.28921336, -0.2122809 ,  0.26400405],
       [-0.7452885 , -0.40291378, -0.25340423,  0.14418639]],
      dtype=float32)>, <tf.Variable 'dense_2/bias:0' shape=(4,) dtype=float32, numpy=array([ 0.00099985,  0.0009998 ,  0.00099986, -0.00099943], dtype=float32)>] 


### initialize from scratch
the variables should be different

In [8]:
network = tf.keras.Sequential([
    tf.keras.layers.InputLayer((None, 4)),
    tf.keras.layers.Dense(4)
    ])

model_2 = BaseModel(network, (None, 2), dir_checkpoint=dir_checkpoint, restore_from_checkpoint=False)
print(model_2.network.weights, "\n\n\n")
print(model_2.train_step, "\n\n\n")
print(model_2.optimizer.variables(), "\n\n\n")

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, None, 4)           20        
                                                                 
Total params: 20
Trainable params: 20
Non-trainable params: 0
_________________________________________________________________
23-02-21 15:21:30 base_model.p WAR   The model can not be saved when it is initialized from scratch with a non-empty checkpoint directory 
[<tf.Variable 'dense_3/kernel:0' shape=(4, 4) dtype=float32, numpy=
array([[ 0.7007702 ,  0.32935625,  0.7594715 , -0.65653986],
       [ 0.33644563, -0.7942023 ,  0.85881466, -0.48365003],
       [ 0.2811125 ,  0.7950495 , -0.6486019 ,  0.4293285 ],
       [ 0.71846217, -0.73578906,  0.381689  ,  0.34892923]],
      dtype=float32)>, <tf.Variable 'dense_3/bias:0' shape=(4,) dtype=float32, numpy=array([0., 0., 0., 0.], dtype=float32)>] 



<tf.

### conflict handling

Trying to save the model creates another checkpoint, since it's a continuation of the original model

In [9]:
model_1.save_model()

23-02-21 15:21:32 base_model.p INF   Successfully saved the model to ./base_checkpoint 


Trying to save the model results in an error, since it was initialized from scratch

In [10]:
model_2.save_model()

Exception: The specified checkpoint directory ./base_checkpoint is not empty, can not save a model initialized from scratch there.